# Initial Lora Adapter

Initial attempts of using LoRA (Low-Rank Adaptation) for model fine-tuning.

In [2]:
''' Loading Dependencies '''
from peft import LoraConfig, TaskType, get_peft_model  # For LoRA fine-tuning
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling  # For loading the pre-trained model and tokenizer
from datasets import Dataset  # For handling the dataset
import torch  # For tensor operations and model training
import pandas as pd  # For loading dataset

/home/jj/repos/DSCI490-Shakespearean-Personality-LLM-Augmentation-/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuring the Model and Adapter

In [3]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Loading the pre-trained model and tokenizer (no quantization for now)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 801.78it/s, Materializing param=model.norm.weight]                              


In [4]:
# LoRA Configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # Type of task (e.g., CAUSAL_LM for language modeling)
    inference_mode=False,  # False for training
    r=8,  # Dimension of the low-rank matricies
    lora_alpha=32,  # Scaling factor for the LoRA updates
    lora_dropout=0.1,  # Dropout rate for the LoRA layers
)

# Adding the adapter to the model
model.add_adapter(lora_config, adapter_name="lora_adapter1")
model.set_adapter("lora_adapter1")

## Loading the Training Data (raw hamlet text)

In [5]:
# Load a pandas df from the text file
hamlet_df1 = pd.read_csv(
	"../data/hamlet_onlyhamletraw.txt",
	sep=r"Ham\.",
	engine="python",
	header=None,
	names=["text"],
)

# Clean invalid rows so tokenizer always receives real text
hamlet_df1 = hamlet_df1.dropna(subset=["text"]).copy()
hamlet_df1["text"] = hamlet_df1["text"].astype(str).str.strip()
hamlet_df1 = hamlet_df1[hamlet_df1["text"] != ""]
print(f"Number of valid rows in the dataset: {len(hamlet_df1)}")

# Set a data collator for language modeling (this will handle padding and masking for us)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Load as a Dataset object
dataset = Dataset.from_pandas(hamlet_df1, preserve_index=False)

# Tokenize the dataset
tokenized_dataset = dataset.map(lambda x: tokenizer(text=x["text"], truncation=True))

Number of valid rows in the dataset: 357


Map: 100%|██████████| 357/357 [00:00<00:00, 7504.96 examples/s]


## Train the Model and Save Output

In [6]:
''' Define the training arguments '''
training_args = TrainingArguments(
    output_dir="../models/lora_finetuned_model",  # Directory to save the model
    num_train_epochs=3,  # Number of training epochs
    per_device_train_batch_size=4,  # Batch size for training
    learning_rate=5e-3,  # Learning rate
    logging_dir="./logs",  # Directory for storing logs
    logging_steps=25,  # Log every 25 steps
)

''' Initialize the Trainer '''
trainer = Trainer(
    model=model,  # The model to train
    args=training_args,  # Training arguments
    train_dataset=tokenized_dataset,  # The tokenized training dataset
    data_collator=data_collator,  # The data collator for language modeling
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [7]:
''' Begin 'training' the model (using LoRA fine-tuning) '''
trainer.train()

Step,Training Loss
25,5.390972
50,6.253301
75,5.887799
100,6.033785
125,5.682673
150,5.850717
175,5.629962
200,5.769343
225,5.566251
250,5.472914


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 49.36it/s]


TrainOutput(global_step=270, training_loss=5.73650543778031, metrics={'train_runtime': 48.3609, 'train_samples_per_second': 22.146, 'train_steps_per_second': 5.583, 'total_flos': 118013110714368.0, 'train_loss': 5.73650543778031, 'epoch': 3.0})

In [8]:
''' Save the adapter '''
model.save_pretrained("../models/lora_finetuned_model1", adapter_name="lora_adapter1")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 46.48it/s]


## Loading the Model & Making Predictions

In [10]:
from peft import PeftModel

# Reload base model + trained LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
inference_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
inference_tokenizer.pad_token = inference_tokenizer.eos_token

inference_model = PeftModel.from_pretrained(
    base_model,
    "../models/lora_finetuned_model1",
    adapter_name="lora_adapter1"
 )
inference_model.eval()

# Simple inference prompt
prompt = "Hamlet reflects on fate and says:"
inputs = inference_tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output_ids = inference_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        repetition_penalty=1.1
    )

print(f"Prompt: {prompt}")
print("Response: ", inference_tokenizer.decode(output_ids[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 869.40it/s, Materializing param=model.norm.weight]                              


Prompt: Hamlet reflects on fate and says:
Response:  Hamlet reflects on fate and says: of th',' but? this you,ll.ll it. the but to what! all'd- his? a this theer you, I. man,, sir to. the thed?ay int the andt.er for that?ed. my play.. the the King.s come in, by. I-es'es; sir beay
